In [ ]:
#jay shree ram

**Fine-tuning Llama 3.2 3B with "openai/gsm8k"**

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf token here"
login(token= HF_TOKEN)

In [ ]:
!pip install -U transformers accelerate

^C


In [ ]:
!pip install -q -U trl transformers accelerate git+https://github.com/huggingface/peft.git
!pip install -q -U datasets bitsandbytes
!pip install trl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00


In [ ]:
import os
import torch
import transformers
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
model_id = 'meta-llama/Llama-3.2-3B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_type = torch.bfloat16
)

In [ ]:
lora_config = LoraConfig(
    r = 16,
    target_modules = ["q_proj", "v_proj"],
    task_type = "CAUSAL_LM"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token= HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config= bnb_config,
                                             device_map= "auto",
                                             token= HF_TOKEN)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
from datasets import load_dataset
dataset = load_dataset('openai/gsm8k', 'main')
data = dataset.map(
    lambda batch: tokenizer(
        [prompt + text for prompt, text in zip(batch['question'], batch['answer'])]
    ),
    batched=True,
    batch_size=32
)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
def formatting_func(example):
  text = f"Question :{example ['prompt']}/nAnswer :{example ['text']}" #[0]}{tokenizer.eos_token}"
  return {"text":text}

In [ ]:
trainer = SFTTrainer(
    train_dataset= data['train'],
    model= model,
    args = transformers.TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 2,
        warmup_steps = 250,
        max_steps = 4000,
        learning_rate = 3e-5,
        bf16 = True,
        fp16 = False,
        logging_steps = 1,
        output_dir = 'output',
        optim = 'paged_adamw_8bit',
        report_to = 'none'
        ),
    peft_config= lora_config,
    formatting_func= formatting_func
    )

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
trainer.train()
# trainer.save_model('output')

Step,Training Loss
1,1.426429
2,1.410053
3,1.102911
4,1.544838
5,1.605115
6,1.342009
7,1.297305
8,1.299453
9,1.491143
10,1.317139


Step,Training Loss
1,1.426429
2,1.410053
3,1.102911
4,1.544838
5,1.605115
6,1.342009
7,1.297305
8,1.299453
9,1.491143
10,1.317139


TrainOutput(global_step=4000, training_loss=1.0564765250384807, metrics={'train_runtime': 12500.0514, 'train_samples_per_second': 0.64, 'train_steps_per_second': 0.32, 'total_flos': 2.1493280737959936e+16, 'train_loss': 1.0564765250384807})

In [ ]:
from transformers import pipeline
from peft import PeftModel

# Reload the base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map='auto',
    token=HF_TOKEN
)

# Load the PEFT adapter weights
FTmodel = PeftModel.from_pretrained(base_model, './output/')

# Merge the adapter weights with the base model
merged_model = FTmodel.merge_and_unload()

In [ ]:
# Change this to your desired repository ID
repo_id = "rajtembe13/Llama-3.2-3B-TUTOR-gsm8k"

# Push the merged model
merged_model.push_to_hub(repo_id)

# Push the tokenizer
tokenizer.push_to_hub(repo_id)

#model metadata
merged_model.config.use_cache = True
merged_model.config.push_to_hub(repo_id)

In [ ]:
# Create a text generation pipeline
pipeline = pipeline("text-generation", model=merged_model, tokenizer=tokenizer)

# Test the fine-tuned model
prompt = "question :what is a^2 * b^2."
sequences = pipeline(
    prompt,
    max_new_tokens=50, # Generate up to 50 new tokens
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")